In [1]:
# Parameters
specific_files = ["Supplementaryfilre.ipynb"]


In [2]:
import numpy as np
import pandas as pd

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import time
import shutil

# ===== 1. ตั้งค่า Path และสร้าง Folder =====
current_dir = os.getcwd()
target_folder = os.path.join(current_dir, "Shop_BKK")

# สร้าง folder shop_BKK ถ้ายังไม่มี
if not os.path.exists(target_folder):
    os.makedirs(target_folder)
    print(f"สร้างโฟลเดอร์: {target_folder}")

# ตั้งค่า Chrome ให้ลงใน workspace หลักก่อน (เพื่อความง่ายในการหาไฟล์ที่เพิ่งโหลด)
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--headless=new") # เพิ่มบรรทัดนี้เพื่อไม่เปิดหน้าต่าง
prefs = {
    "download.default_directory": current_dir,
    "download.prompt_for_download": False,
    "safebrowsing.enabled": True
}
chrome_options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(options=chrome_options)

try:
    # ===== 2. Login และไปหน้าจัดการร้านค้า =====
    driver.get("https://cpi.moc.go.th/cpi/login.aspx")
    
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "txtUser")))
    driver.find_element(By.ID, "txtUser").send_keys("index_bkk027")
    driver.find_element(By.ID, "txtPwd").send_keys("Tpsocpi1234")
    driver.find_element(By.ID, "btnLogin").click()

    WebDriverWait(driver, 20).until_not(EC.url_contains("login.aspx"))
    driver.get("https://cpi.moc.go.th/cpi/shopMgt.aspx")

    # ===== 3. สั่งดาวน์โหลด =====
    btn_excel = WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable((By.ID, "ContentPlaceHolder1_btnExcel"))
    )
    
    # บันทึกรายชื่อไฟล์ก่อนดาวน์โหลด เพื่อเปรียบเทียบหาไฟล์ใหม่
    files_before = set(os.listdir(current_dir))
    btn_excel.click()
    print("📂 กำลังดาวน์โหลดไฟล์...")

    # ===== 4. รอจนดาวน์โหลดเสร็จ (Check loop) =====
    downloaded_file = None
    seconds_waited = 0
    while seconds_waited < 30: # รอสูงสุด 30 วินาที
        time.sleep(1)
        files_after = set(os.listdir(current_dir))
        new_files = files_after - files_before
        
        # กรองเอาเฉพาะไฟล์ .xlsx และไม่ใช่ไฟล์ชั่วคราว (.crdownload)
        target_files = [f for f in new_files if f.endswith(".xlsx") and not f.startswith("~")]
        
        if target_files:
            downloaded_file = target_files[0]
            break
        seconds_waited += 1

    # ===== 5. เปลี่ยนชื่อและย้ายไฟล์ =====
    if downloaded_file:
        old_path = os.path.join(current_dir, downloaded_file)
        new_path = os.path.join(target_folder, "shop_data.xlsx")
        
        # หากมีไฟล์ชื่อเดิมอยู่แล้วให้ลบก่อน (หรือจะเขียนทับก็ได้)
        if os.path.exists(new_path):
            os.remove(new_path)
            
        shutil.move(old_path, new_path)
        print(f"✅ ย้ายและเปลี่ยนชื่อไฟล์สำเร็จ: {new_path}")
    else:
        print("❌ หาไฟล์ที่ดาวน์โหลดไม่เจอ หรือดาวน์โหลดนานเกินไป")

finally:
    driver.quit()

📂 กำลังดาวน์โหลดไฟล์...


✅ ย้ายและเปลี่ยนชื่อไฟล์สำเร็จ: c:\Users\User\Documents\GitHub\AutomationCPI\01-database\Shop_BKK\shop_data.xlsx
